# zolt — zolt.ai | Notebook de Treino

**Coding-Agent e Reasoning AI | 250M parâmetros (com sub-rede zolt-mini) | e4b / MatFormer**

Este notebook cobre:
1. Setup do ambiente (GPU check)
2. Clone do repositório zolt
3. Download de dados (StarCoderData)
4. Filtro e tokenização
5. Treino do tokenizer BPE
6. Loop de treino 250M
7. Avaliação

> **Compute recomendado**: RunPod RTX 4090 / A100 ou Colab Pro+

## 0. Verificar GPU

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA disponível: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    print(f'BF16 suportado: {torch.cuda.is_bf16_supported()}')
else:
    print('⚠ Sem GPU — o treino será muito lento. Recomenda-se RunPod/Colab Pro.')

## 1. Instalação

In [ ]:
# Instalar dependências
!pip install -q tokenizers datasets einops tqdm wandb

# Clone do repositório zolt (substituir pelo URL real)
# !git clone https://github.com/SEU_USER/zolt.git
# %cd zolt

# OU: se já tens o código no Drive/RunPod, ajusta o path:
import sys
# sys.path.insert(0, '/content/zolt')  # Colab
# sys.path.insert(0, '/workspace/zolt')  # RunPod
sys.path.insert(0, '.')

## 2. Smoke Test da Arquitectura

In [ ]:
from zolt.config import ZoltConfig
from zolt.model import ZoltForCausalLM
import torch

config = ZoltConfig()
model = ZoltForCausalLM(config)
n_params = sum(p.numel() for p in model.parameters())
print(f'zolt parâmetros: {n_params:,} ({n_params/1e6:.1f}M)')

# Forward pass rápido
ids = torch.randint(0, config.vocab_size, (2, 64))
logits, loss = model(ids, labels=ids)
print(f'Forward OK | logits: {logits.shape} | loss: {loss.item():.3f}')

## 3. Download de Dados (StarCoderData)

In [ ]:
import os
# Opcional: HuggingFace token (para Stack v2 — StarCoder não precisa)
# os.environ['HF_TOKEN'] = 'hf_xxx'

# Download StarCoderData (sem gating, acesso livre)
!python -m zolt.data.download \
    --source starcoder \
    --output_dir data/raw \
    --langs javascript typescript python vue \
    --max_samples 300000

## 4. Filtrar Dados

In [ ]:
!python -m zolt.data.pipeline filter \
    --raw_dir data/raw \
    --filtered_dir data/filtered

## 5. Treinar Tokenizer BPE (32K)

In [ ]:
!python -m zolt.tokenizer.train_tokenizer \
    --data_dirs data/filtered \
    --output zolt_tokenizer.json \
    --vocab_size 32000

## 6. Tokenizar Dados → .bin

In [ ]:
!python -m zolt.data.pipeline tokenize \
    --filtered_dir data/filtered \
    --tokenizer zolt_tokenizer.json \
    --tokens_dir data/tokens

# Verificar total de tokens
!python -m zolt.data.pipeline validate --tokens_dir data/tokens

## 7. Treino zolt 250M

In [ ]:
import glob
token_files = ' '.join(glob.glob('data/tokens/*.bin'))
print(f'Ficheiros de tokens: {token_files}')

# Configuração recomendada para RTX 4090 (24GB VRAM)
# batch_size=8 + grad_accum=4 = effective batch 32
# ~100K steps ≈ 3B tokens

!python -m zolt.train \
    --token_files {token_files} \
    --output_dir checkpoints/ \
    --max_seq_len 4096 \
    --batch_size 8 \
    --grad_accum 4 \
    --lr 3e-4 \
    --lr_min 3e-5 \
    --warmup_steps 500 \
    --total_steps 100000 \
    --save_every 2000 \
    --log_every 50 \
    --dtype bf16 \
    --wandb

## 8. Avaliação de Checkpoint

In [ ]:
import glob
checkpoints = sorted(glob.glob('checkpoints/ckpt-step*'))
latest = checkpoints[-1] if checkpoints else None
print(f'Último checkpoint: {latest}')

if latest:
    !python -m zolt.eval \
        --checkpoint {latest} \
        --eval_jsonl data/eval.jsonl

## 9. Extensão de Contexto 4K → 16K (Fase 3)

In [ ]:
# Após o treino base estar estável, aplicar RoPE NTK scaling
import glob
checkpoints = sorted(glob.glob('checkpoints/ckpt-step*'))
best_ckpt = checkpoints[-1]  # ou escolhe manualmente pelo loss

!python -m zolt.rope_scaling \
    --checkpoint {best_ckpt} \
    --output checkpoints/zolt-16k \
    --target_len 16384 \
    --method ntk

print('zolt-16k pronto para fine-tuning de extensão de contexto!')

## Referências

- [The Stack v2](https://huggingface.co/datasets/bigcode/the-stack-v2) — aceitar termos antes de usar
- [StarCoderData](https://huggingface.co/datasets/bigcode/starcoderdata) — sem gating
- [NTK-aware RoPE scaling](https://arxiv.org/abs/2309.16039)
- [Chinchilla scaling laws](https://arxiv.org/abs/2203.15556)
- [Gemma 3n / MatFormer](https://arxiv.org/abs/2310.06694)